# CEMA Trace Capture - FPGA & Microcontroller


**Author:** Dev Mehta

**Description:** Unified capture workflow supporting both CW305 FPGA and CW Lite microcontroller targets.

## Imports and Parameters

In [ ]:
# Imports
import chipwhisperer as cw
import sys
import time
import numpy as np
from datetime import datetime
from tqdm.notebook import trange

#adjust scapegoat path based on your local setup
sys.path.insert(1, '../../SCApeGoat-main/')

from WPI_SCA_LIBRARY.CWScope import *
from WPI_SCA_LIBRARY.LeakageModels import *
%run "../function/CEMA_functions.ipynb"

# Global Parameters
N_traces = 10000
no_samples = 10400
clk_spd = 5E6
vcc_in = 1.0

## Hardware Selection
Choose target platform: FPGA (CW305) or Microcontroller (CW Lite)

## Setup Firmware (Microcontroller Only)

In [ ]:
# ===== SELECT TARGET =====
# Set TARGET to either "FPGA" or "UC" (microcontroller)
TARGET = "FPGA"  # Options: "FPGA" or "UC"

# Probe diameter is the same for all platforms
probe_diameter = "1.2mm"

if TARGET == "FPGA":
    # ===== CW305 FPGA Configuration =====
    # Bitfile options:
    #   - "impl_3.bit"          
    #   - None - bitfile (default: standard CW AES implementation)
    bitfile = "impl_3.bit"
    
    # Motor grid parameters (FPGA-specific side length)
    s_len = 2878800  # stepper motor side length in steps
    n_steps = 10     # grid points per axis (10x10 grid)
    stepsize = -s_len // n_steps
    
    dut = "CW305"
    
    print(f"Target: FPGA (CW305)")
    print(f"Bitfile: {bitfile}")
    print(f"Probe diameter: {probe_diameter}")
    print(f"Grid: {n_steps}x{n_steps}, stepsize: {stepsize}")

elif TARGET == "UC":
    # ===== CW Lite Microcontroller Configuration =====
    # SimpleSerial target (no bitfile needed)
    bitfile = None  # Not used for microcontroller
    
    # Motor grid parameters (UC-specific side length)
    s_len = 160000 * 12  # stepper motor side length in steps
    n_steps = 10          # grid points per axis (10x10 grid)
    stepsize = -s_len // n_steps
    
    dut = "CW-Lite"
    
    print(f"Target: Microcontroller (CW Lite)")
    print(f"Probe diameter: {probe_diameter}")
    print(f"Grid: {n_steps}x{n_steps}, stepsize: {stepsize}")

else:
    raise ValueError(f"Unknown TARGET: {TARGET}. Choose 'FPGA' or 'UC'.")

In [ ]:
if TARGET == "UC":
    # ===== Run Setup Script for Microcontroller Firmware =====
    # This compiles and loads the firmware to the CW Lite target
    %run "../function/Setup_script.ipynb"
    print(f"Firmware compiled and loaded to CW Lite")
else:
    print(f"Skipping firmware setup (FPGA target)")

## Setup Firmware and Scope

In [ ]:
if TARGET == "FPGA":
    # ===== FPGA Setup (CW305) =====
    if bitfile is None:
        print(f"Using default CW305 AES bitfile")
        scope_cw = CWScope(
        None,
        target_type=cw.targets.CW305,
        fpga_id='100t',
        force=True
        )
    else:
        print(f"Using default CW305 AES bitfile")
        scope_cw = CWScope(
        None,
        target_type=cw.targets.CW305,
        fpga_id='100t',
        force=True,
        bsfile=bitfile
        )
    print(f"CW305 FPGA connected with bitfile: {bitfile}")

elif TARGET == "UC":
    # ===== Microcontroller Setup (CW Lite) =====
    # Firmware already loaded by Setup_script.ipynb; now connect scope and target
    scope_cw = CWScope(
        fw_path,
        target_type=cw.targets.SimpleSerial,
        target_programmer=cw.programmers.STM32Programmer
    )
    print(f"CW Lite microcontroller connected")

## Hardware Configuration

In [ ]:
if TARGET == "FPGA":
    # ===== PLL Configuration (FPGA-specific) =====
    scope_cw.target.vccint_set(vcc_in)
    
    # Enable PLL and configure output clocks
    scope_cw.target.pll.pll_enable_set(True)            # Enable PLL chip
    scope_cw.target.pll.pll_outenable_set(False, 0)    # Disable unused PLL0
    scope_cw.target.pll.pll_outenable_set(True, 1)     # Enable PLL output 1
    scope_cw.target.pll.pll_outenable_set(False, 2)    # Disable unused PLL2
    scope_cw.target.pll.pll_outfreq_set(clk_spd, 1)    # Set clock frequency
    
    # USB and sleep configuration
    scope_cw.target.clkusbautooff = True
    scope_cw.target.clksleeptime = 1  # 1ms idle time
    
    print("PLL configured")

# ===== ADC Configuration (common to both targets, FPGA uses multiplier) =====
scope_cw.scope.adc.samples = no_samples
scope_cw.scope.clock.clkgen_src = "extclk"

if TARGET == "FPGA":
    scope_cw.scope.clock.adc_mul = 40  # FPGA: high multiplier for oversampling
elif TARGET == "UC":
    scope_cw.scope.clock.adc_mul = 1   # UC: lower multiplier (adjust as needed)

print(f"ADC samples: {no_samples}")
print(f"ADC multiplier: {scope_cw.scope.clock.adc_mul}")

# Verify ADC lock
assert scope_cw.scope.clock.adc_locked, "ADC failed to lock"

# Set gain
scope_cw.scope.gain.db = 15

# Disable low-gain errors (optional safety setting)
scope_cw.scope.adc.lo_gain_errors_disabled = False

print("Hardware configuration complete")

## Motor Setup

In [ ]:
# Initialize XYZ motorized stage
A = XYZ()
X, Y, Z, interface = A.XYZ_setup(velocity=10000, acceleration=10000)

print(f"Motor stage initialized")
print(f"Grid definition:")
print(f"  - Side length: {s_len} steps")
print(f"  - Grid size: {n_steps}x{n_steps}")
print(f"  - Step size: {stepsize} steps")

## Manual Motor Control (Optional)
Uncomment the examples below to manually position the probe for alignment or testing.

In [ ]:
# ===== Manual Motor Controls to Position Probe =====
# Uncomment and run to manually move to the top-left corner
# before starting automated grid capture. Useful for probe alignment and testing.
# 
# --- Get current position ---
# x_current = X.get_actual_position()  # Returns current X position in steps
# y_current = Y.get_actual_position()  # Returns current Y position in steps
# z_current = Z.get_actual_position()  # Returns current Z position in steps
# print(f"X: {x_current}, Y: {y_current}, Z: {z_current}")
# 
# --- Move by relative amount (relative to current position) ---
# X.move_by(-500000)   # Move X left by 500000 steps (negative = left)
# Y.move_by(-500000)   # Move Y up by 500000 steps (negative = up)
# Z.move_by(100000)    # Move Z up by 100000 steps (positive = up)
# 
# --- Move to specific absolute position ---
# x_target = -1000000  # Target X position in steps
# y_target = -1000000  # Target Y position in steps
# X.move_to(x_target)  # Move X axis to absolute position
# Y.move_to(y_target)  # Move Y axis to absolute position
# 
# --- Continuous rotation (for Z axis probe movement) ---
# Z.rotate(500000)     # Rotate Z continuously by 500000 steps (positive = rotate)
# Z.stop()             # Stop rotation immediately
# 
# Example: Position probe at top-left corner
# x_target = -s_len // 2   # Move to left edge
# y_target = -s_len // 2   # Move to top edge
# X.move_to(x_target)
# Y.move_to(y_target)
# Z.move_by(100000)        # Lift probe slightly
# print(f"Probe positioned at top-left corner")

## Create Experiment and Add Metadata

In [ ]:
# Initialize SCApeGoat experiment manager
emsca = FileParent("CEMA", "./", True)

# Create timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Build experiment name encoding key parameters
mode = "raw"  # 'raw' or 'avg' depending on post-processing preference
experiment_name = f"{TARGET}_{probe_diameter.replace('mm', '')}mm_{N_traces}_{n_steps}x{n_steps}_{N_traces}_traces"

# Create experiment
test = emsca.add_experiment(experiment_name)

# Add metadata
test.update_metadata("Author", "User")
test.update_metadata("#traces", str(N_traces))
test.update_metadata("probe_diameter", probe_diameter)
test.update_metadata("filtered", "yes")
test.update_metadata("grid_size", str(n_steps))
test.update_metadata("dut", dut)
test.update_metadata("side_length", str(s_len))
test.update_metadata("date", timestamp)
test.update_metadata("clean_supply", "yes")
test.update_metadata("capture", "cw_husky")
test.update_metadata("target", "AES_unmasked")
test.update_metadata("grid", f"{n_steps}x{n_steps}")
test.update_metadata("sampling_rate", str(scope_cw.scope.clock.adc_mul))

if TARGET == "FPGA":
    test.update_metadata("bitfile", bitfile)
    test.update_metadata("platform", "FPGA_CW305")
elif TARGET == "UC":
    test.update_metadata("platform", "Microcontroller_CWLite")

print(f"Experiment created: {experiment_name}")
print(f"Metadata initialized")

## Load Plaintext/Key Data
Load from pre-generated 5k or 5kx10 plaintext/key experiments for consistency with analysis notebooks.

In [ ]:
# Load plaintext/key experiments
# Uses pre-generated 5k and 5kx10 plaintext/key sets for consistency with analysis notebooks
try:
    # Try to load the larger 5kx10 experiment first (50k traces)
    try:
        pt = emsca.get_experiment("pt_keys_5kx10")
        pt_name = "pt_keys_5kx10"
        print(f"Using pt_keys_5kx10 experiment (50k plaintext/key pairs)")
    except:
        # Fall back to standard 5k experiment
        pt = emsca.get_experiment("pt_keys_5k")
        pt_name = "pt_keys_5k"
        print(f"Using pt_keys_5k experiment (5k plaintext/key pairs)")
    
    # keys_pt = pt.get_dataset("keys").read_data(0, N_traces)
    # fixed_pt = pt.get_dataset("fixed_pt").read_data(0, N_traces)
    # random_pt = pt.get_dataset("plaintexts").read_data(0, N_traces)
    # print(f"Successfully loaded plaintext/key data from experiment: {pt_name}")
except Exception as e:
    print("ERROR: Could not load plaintext/key experiments.")
    print("Please ensure either 'pt_keys_5kx10' or 'pt_keys_5k' experiment exists.")
    print("Both experiments must have: keys, fixed_pt, and plaintexts datasets.")
    raise

## Capture Traces
Run grid-based trace capture across the motorized XY positions.

In [ ]:
# Execute grid tracing capture
# Captures N_traces at each grid position (i,j)
# Stores fixed_i_j and random_i_j datasets in the experiment

print(f"Starting trace capture on {n_steps}x{n_steps} grid...")
print(f"Total grid positions: {(n_steps)*(n_steps)}")
print(f"Traces per position: {N_traces}")
print(f"Total traces: {(n_steps)*(n_steps)*N_traces}")

Grid_Tracing_scapegoat(
    stepsize,
    stepsize,
    n_steps,
    n_steps,
    X,
    Y,
    Z,
    interface,
    scope_cw,
    pt,
    test,
    N_traces
)

print(f"\nCapture complete!")
print(f"Experiment: {test.name}")
print(f"Datasets stored in SCApeGoat FileParent.")

## Disconnect and Clean Up

In [ ]:
# Disconnect scope and target
try:
    scope_cw.disconnect()
    print("Scope disconnected successfully")
except:
    print("Warning: Could not cleanly disconnect scope")

print("\n" + "="*60)
print(f"Capture workflow complete for {TARGET}")
print(f"Experiment: {test.name}")
print(f"Location: SCApeGoat FileParent at ./CEMA/")
print("\nNext step: Run analysis notebook to compute metrics (TVLA, CEMA, SNR).")
print("="*60)